Họ và tên: Nguyễn Đình Khanh

MSSV: 24110244

GitHub Link: https://github.com/qilskcter/TriTueNhanTao

In [8]:
import numpy as np

dirty      =  1
clean     =  0
obstacle  = -1

suck   = 'SUCK'
up  = 'UP'
down = 'DOWN'
left  = 'LEFT'
right  = 'RIGHT'

In [23]:
def create_ground(row, col):
    ground = np.zeros((row, col), dtype=int)
    for i in range(row):
        for j in range(col):
            r = np.random.random()
            if r < 0.35:
                ground[i][j] = dirty
            elif r < 0.55:
                ground[i][j] = obstacle
    ground[0][0] = clean
    has_dirty = False
    for i in range(row):
        for j in range(col):
            if ground[i][j] == dirty:
                has_dirty = True
    if not has_dirty:
        ground[0][1] = dirty
    return ground

In [10]:
def create_map(row, col):
    memory = []
    for i in range(row):
        row = []
        for j in range(col):
            row.append('unknown')
        memory.append(row)
    return memory

In [11]:
def feel(ground, index):
    x, y = index
    if ground[x][y] == dirty:
        state = 'dirty'
    elif ground[x][y] == obstacle:
        state = 'obstacle'
    else:
        state = 'clean'
    return {'index': index, 'state': state}

In [12]:
def update(memory, percept):
    x, y = percept['index']
    memory[x][y] = percept['state']
    return memory   

In [13]:
def moveable(memory, x, y, row, col):
    if x < 0 or x >= row:
        return False
    if y < 0 or y >= col:
        return False
    if memory[x][y] == 'obstacle':
        return False
    return True

In [24]:
def decide_action(percept, memory, rows, cols):
    if percept['state'] == 'dirty':
        return suck

    all_clear = True
    for i in range(rows):
        for j in range(cols):
            if memory[i][j] in ('unknown', 'dirty'):
                all_clear = False
    if all_clear:
        return None

    x, y = percept['index']
    if x % 2 == 0:
        directions = [right, down, left, up]
    else:
        directions = [left, down, right, up]

    moves = {right: (0, 1), left: (0, -1), down: (1, 0), up: (-1, 0)}

    for direction in directions:
        dx, dy = moves[direction]
        if moveable(memory, x + dx, y + dy, rows, cols):
            return direction

    return None

In [25]:
def perform_action(action, ground, position, memory):
    x, y = position
    if action == suck:
        ground[x][y] = clean
        memory[x][y] = 'clean'
    elif action == up:   position = (x - 1, y)
    elif action == down: position = (x + 1, y)
    elif action == left: position = (x, y - 1)
    elif action == right: position = (x, y + 1)
    return position, ground, memory

In [26]:
def robot(ground, position, memory, rows, cols):
    percept = feel(ground, position)
    memory = update(memory, percept)
    action = decide_action(percept, memory, rows, cols)
    return action, memory

In [29]:
rows = 3
cols  = 3

ground = create_ground(rows, cols)
position = (0, 0)
memory = create_map(rows, cols)
step = 0

print('San nha (0=sach, 1=ban, -1=co vat can):')
print(np.array(ground))
print(f'\nBat dau ta: {list(position)}\n')

while step < 100:
    action, memory = robot(ground, position, memory, rows, cols)

    if action is None:
        print(f'Buoc {step + 1}: {list(position)} -> Dung lai')
        break

    print(f'Buoc {step + 1}: {list(position)} -> {action}')
    position, ground, memory = perform_action(action, ground, position, memory)
    step += 1

print(np.array(ground))

remaining_dirty = 0
for i in range(rows):
    for j in range(cols):
        if ground[i][j] == dirty:
            remaining_dirty += 1

if remaining_dirty == 0:
    print(f'San nha sach sau {step} buoc')
else:
    print(f'{remaining_dirty} o ban con lai bi khoa boi vat can.')

San nha (0=sach, 1=ban, -1=co vat can):
[[ 0  0  0]
 [ 1  1  1]
 [ 1  1 -1]]

Bat dau ta: [0, 0]

Buoc 1: [0, 0] -> RIGHT
Buoc 2: [0, 1] -> RIGHT
Buoc 3: [0, 2] -> DOWN
Buoc 4: [1, 2] -> SUCK
Buoc 5: [1, 2] -> LEFT
Buoc 6: [1, 1] -> SUCK
Buoc 7: [1, 1] -> LEFT
Buoc 8: [1, 0] -> SUCK
Buoc 9: [1, 0] -> DOWN
Buoc 10: [2, 0] -> SUCK
Buoc 11: [2, 0] -> RIGHT
Buoc 12: [2, 1] -> SUCK
Buoc 13: [2, 1] -> RIGHT
Buoc 14: [2, 2] -> Dung lai
[[ 0  0  0]
 [ 0  0  0]
 [ 0  0 -1]]
San nha sach sau 13 buoc
